In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.base import clone

import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

import os
from tqdm import tqdm
import random

from prepare_data import prepare_data
from preprocessor_function import preprocessor_function

In [2]:
#объявляем random.seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
#загружаем тренировочные данные
data = pd.read_csv('data/train.csv')
data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [4]:
y = np.log1p(data['SalePrice'])
X = prepare_data(data.drop('SalePrice',axis=1))

In [5]:
#удаляем выбросы
X = X[X['GrLivArea'] < 4500] 

In [6]:
#синхронизируем данные
y = y[X.index]

In [7]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(0.2/0.9), random_state=SEED)

In [8]:
numeric_colums = X_train.select_dtypes(include=['number']).columns
categorical_colums = X_train.select_dtypes(include=['object', 'string']).columns

In [9]:
preprocessor = preprocessor_function(X_train, categorical_colums, numeric_colums)

In [ ]:
preview_preprocessor = clone(preprocessor).set_output(transform='pandas')
X_encoded_preview = preview_preprocessor.fit_transform(X, y)
X_encoded_preview.head()

,Street_Pave,Alley_Grvl,Alley_Pave,LotShape_IR2,LotShape_IR3,LotShape_Reg,LandContour_HLS,LandContour_Low,LandContour_Lvl,Utilities_NoSeWa,...,PoolArea,MiscVal,MoSold,YrSold,HouseAge,RemodAge,TotalBath,HasGarage,HasBasement,TotalSF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,-1.601578,0.138375,-1.045249,-0.871676,1.174852,0.242536,0.161363,0.011436
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,-0.490155,-0.614427,-0.185182,0.388660,0.386571,0.242536,0.161363,-0.042838
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,0.991743,0.138375,-0.979090,-0.823201,1.174852,0.242536,0.161363,0.192351
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,-1.601578,-1.367230,1.799589,0.631032,-1.189990,0.242536,0.161363,-0.108743
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,2.103167,0.138375,-0.946011,-0.726252,1.174852,0.242536,0.161363,1.015514


In [11]:
generator = torch.Generator().manual_seed(SEED)

# Применяем препроцессор
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Преобразуем данные в тензоры PyTorch
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Создаем TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Создаем DataLoader'ы
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True, generator=generator)
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

c:\Users\angelika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [8, 9, 14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\angelika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [ ]:
#создаем модель 
class MyNN(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input,128)
        self.batch_norm_1 = nn.BatchNorm1d(128)
        self.act_1 = nn.LeakyReLU()
        self.dropout_1 = nn.Dropout(0.4)
        self.layer_2 = nn.Linear(128,64)
        self.batch_norm_2 = nn.BatchNorm1d(64)
        self.act_2 = nn.LeakyReLU()
        self.dropout_2 = nn.Dropout(0.2)
        self.layer_3 = nn.Linear(64,32)
        self.batch_norm_3 = nn.BatchNorm1d(32)
        self.act_3 = nn.ReLU()
        self.dropout_3 = nn.Dropout(0.3)
        self.layer_4 = nn.Linear(32,output)

    def forward(self,X):
        X = self.layer_1(X)
        X = self.batch_norm_1(X)
        X = self.act_1(X)
        X = self.dropout_1(X)
        X = self.layer_2(X)
        X = self.batch_norm_2(X)
        X = self.act_2(X)
        X = self.dropout_2(X)
        X = self.layer_3(X)
        X = self.batch_norm_3(X)
        X = self.act_3(X)
        X = self.dropout_3(X)
        out = self.layer_4(X)
        return out

In [24]:
#объявляем модель
model = MyNN(X_train_processed.shape[1], 1)

In [25]:
# задаем оптимизатор и метрику
loss_reg = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=1, weight_decay=0.01)

In [26]:
# задаем шедулер
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,  #оптимизатор
                                                          mode = 'min',  #'max' или 'min' - следим чтобы отслеживаемый параметр увеличивался()
                                                          factor = 0.1,  #коэфициэнт, на который будет умножен lr
                                                          patience = 10, #кол-во эпох без улучшения отслеживаемого пораметров
                                                          eps = 1e-8,
                                                          )

In [ ]:
#начинаем процесс обучения
EPOCHS = 500
train_loss = []
val_loss = []
lr_list = []
best_loss = None
best_model = 0
count = 0

for epoch in range(EPOCHS):
    model.train()
    train_loop = tqdm(train_loader,leave=False)
    running_train_loss = []
    for x,targets in train_loop:

        pred = model(x)
        loss = torch.sqrt(loss_reg(pred, targets))

        opt.zero_grad()
        loss.backward()

        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        train_loop.set_description(f'Epoch[{epoch+1}/{EPOCHS}], train_loss = {mean_train_loss:.4f}')

    train_loss.append(mean_train_loss)

    model.eval()
    with torch.no_grad():
        running_val_loss = []
        for x, targets in val_loader:
            pred = model(x)
            loss = torch.sqrt(loss_reg(pred,targets))

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

        val_loss.append(mean_val_loss)

    lr_scheduler.step(mean_val_loss)

    lr = lr_scheduler._last_lr[0]
    lr_list.append(lr)
    print((f'Epoch [{epoch+1}/{EPOCHS}], train_loss = {mean_train_loss:.4f},  val_loss = {mean_val_loss:.4f}, lr = {lr:.4f}'))

    if best_loss is None:
      best_loss = mean_val_loss

    if mean_val_loss < best_loss:
      best_loss = mean_val_loss
      count = 0
      checkpoint = {
          'state_model': model.state_dict(),
          'state_opt' : opt.state_dict(),
          'state_lr_scheduler': lr_scheduler.state_dict(),
          'loss' : {
              'train_loss':train_loss,
              'val_loss': val_loss,
              'best_loss': best_loss,
          },
          'metric':{
              'train_loss' : train_loss,
              'val_loss' : val_loss,
          },
          'lr' : lr_list,
          'epoch' : {
              'EPOCHS' : EPOCHS,
              'save_epoch': epoch
          }
      }

      if best_model == 0:
        torch.save(model.state_dict(), f'best_nn_model/model_state_dict_epoch_{epoch+1}.pt')
        best_model = epoch+1

      else:
        os.remove(f'best_nn_model/model_state_dict_epoch_{best_model}.pt')
        torch.save(model.state_dict(), f'best_nn_model/model_state_dict_epoch_{epoch+1}.pt')
        best_model = epoch+1

      print(f'На эпохе - {epoch+1}, сохранена модель со значением функции потерь на валидации - {best_loss:.4f}', end='\n\n') 

    if count >= 150:
      print(f'\033[31Обучение остановлено на {epoch+1} эпохе.\033[0m')
      break
    count += 1

Epoch [1/500], train_loss = 3.4462,  val_loss = 1.1732, lr = 1.0000


Epoch [2/500], train_loss = 0.4164,  val_loss = 0.3034, lr = 1.0000
На эпохе - 2, сохранена модель со значением функции потерь на валидации - 0.3034



Epoch [3/500], train_loss = 0.3683,  val_loss = 0.6232, lr = 1.0000


Epoch [4/500], train_loss = 0.3704,  val_loss = 0.4597, lr = 1.0000


Epoch [5/500], train_loss = 0.3470,  val_loss = 0.4124, lr = 1.0000


...

Epoch [200/500], train_loss = 0.2624,  val_loss = 0.2745, lr = 0.0000


Epoch [201/500], train_loss = 0.2351,  val_loss = 0.2690, lr = 0.0000


Epoch [202/500], train_loss = 0.2539,  val_loss = 0.2712, lr = 0.0000


Epoch [203/500], train_loss = 0.2434,  val_loss = 0.2663, lr = 0.0000


Epoch [204/500], train_loss = 0.2383,  val_loss = 0.2767, lr = 0.0000
[31Обучение остановлено на 204 эпохе.


In [ ]:
# загружаем веса лучшей модели и начинаем процесс тестирования
model.load_state_dict(torch.load(f'best_nn_model/model_state_dict_epoch_{best_model}.pt'))

model.eval()
with torch.no_grad():
    running_test_loss = []
    for x, targets in test_loader:
        pred = model(x)
        loss = torch.sqrt(loss_reg(pred,targets))
        running_test_loss.append(loss.item())

    mean_test_loss = sum(running_test_loss)/len(running_test_loss)
    print(f'Финальный loss = {mean_test_loss:.4f}')

Финальный loss = 0.4210


In [ ]:
# 2. Загружаем тестовый набор данных 
test_data = pd.read_csv('data/test.csv')
test_ids = test_data['Id']

# 3. Предобработка тестовых данных
X_test_final = prepare_data(test_data)
X_test_processed_final = preprocessor.transform(X_test_final)

# Конвертируем в тензор
X_test_tensor_final = torch.tensor(X_test_processed_final, dtype=torch.float32)

print("Данные готовы к предсказанию.")

Данные готовы к предсказанию.


c:\Users\angelika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 8, 11] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [ ]:
model.eval()
with torch.no_grad():
    # Получаем предсказания
    log_preds = model(X_test_tensor_final).numpy().flatten()

# Переводим из логарифмического масштаба обратно
final_preds = np.expm1(log_preds)

# Создаем DataFrame для отправки
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_preds
})

# Сохраняем в CSV
submission.to_csv('data/nn_submission/submission.csv', index=False)

print("Файл submission.csv успешно создан!")
display(submission.head(30))

Файл submission.csv успешно создан!


,Id,SalePrice
0,1461,131898.796875
1,1462,155031.093750
2,1463,165267.937500
3,1464,171356.281250
4,1465,169083.781250
5,1466,162941.546875
6,1467,164827.843750
7,1468,156904.187500
8,1469,171125.046875
9,1470,131898.796875
